# NBA Analytics — Train Models on Google Colab

This notebook trains your NBA models using Google Colab's cloud hardware.

**How to use:**
1. Open this notebook in VS Code
2. Click **Select Kernel** (top right) → choose **Colab** → pick **T4 GPU**
3. **Before running**, create a data zip on your local machine (see Step 2)
4. Run the cells in order

## Step 1: Set up the environment

In [1]:
# Verify GPU is available
!nvidia-smi

Sun Mar  1 17:26:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.60                 Driver Version: 581.60         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro RTX 3000              WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   54C    P8             10W /   80W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import sys

REPO_URL = "https://github.com/SpencerGoss/nba-analytics-project.git"
PROJECT_DIR = "/content/nba-analytics-project"

# Clone the repo (code only — data files are gitignored)
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print(f"Repo already cloned at {PROJECT_DIR}")

# Set working directory so relative paths in model code work
os.chdir(PROJECT_DIR)

# Add project root to Python path so 'from src.models...' imports work
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Install dependencies
!pip install -q -r requirements.txt

# Create output directories
os.makedirs("models/artifacts", exist_ok=True)

print(f"\nWorking directory: {os.getcwd()}")
print(f"Project files: {sorted(os.listdir('.'))}")

Cloning into '/content/nba-analytics-project'...



Working directory: c:\content\nba-analytics-project
Project files: ['.claude', '.git', '.gitignore', 'README.md', 'backfill.py', 'docs', 'models', 'reports', 'requirements.txt', 'scripts', 'src', 'update.py']


'pip' is not recognized as an internal or external command,
operable program or batch file.


## Step 2: Upload your data files

Data files are gitignored and not included in the clone, so you need to upload them.

**On your local machine**, open a terminal in your project folder and run:

```
cd C:\Users\spenc\OneDrive\Desktop\GIT\nba-analytics-project
python -c "import shutil; shutil.make_archive('nba_data', 'zip', '.', 'data')"
```

This creates `nba_data.zip` containing your entire `data/` folder. Then run the cell below to upload it.

In [3]:
import os
import zipfile
from google.colab import files

print("Select your nba_data.zip file...")
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall(".")
        print(f"Extracted {fname}")
        os.remove(fname)

# Verify the files the models need
required = [
    "data/features/game_matchup_features.csv",
    "data/features/player_game_features.csv",
    "data/features/team_game_features.csv",
    "data/processed/standings.csv",
    "data/processed/team_game_logs.csv",
]
print("\nData file check:")
all_ok = True
for f in required:
    exists = os.path.isfile(f)
    status = "OK" if exists else "MISSING"
    print(f"  {status:>7}  {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll required files present. Ready to train!")
else:
    print("\nSome files are missing. Rebuild features in Step 3 or re-upload.")

ModuleNotFoundError: No module named 'google'

## Step 3: Rebuild features (optional)

Run this cell **only** if you need to recompute your feature tables.
Skip it if the data file check above shows all files present.

In [ ]:
# OPTIONAL: Rebuild feature tables from processed data
# Uncomment and run if you need fresh features

# from src.features.team_game_features import build_team_game_features, build_matchup_dataset
# from src.features.player_features import build_player_game_features

# build_team_game_features()
# build_matchup_dataset()
# build_player_game_features()

## Step 4: Train the models

This runs all three training tasks:
1. **Game Outcomes** — predicts who wins each game
2. **Player Performance** — predicts individual player stats
3. **Playoff Odds** — simulates playoff chances for each team

In [ ]:
from datetime import datetime

from src.models.game_outcome_model import train_game_outcome_model
from src.models.player_performance_model import train_player_models
from src.models.playoff_odds_model import simulate_playoff_odds

start = datetime.now()
print("=" * 60)
print("STARTING MODEL TRAINING ON COLAB")
print("=" * 60)

In [ ]:
# Task 1: Game Outcome Model
print("\nTraining Game Outcome Model...")
print("-" * 40)
game_model, game_metrics = train_game_outcome_model()
print(f"\nDone! Accuracy: {game_metrics.get('test_accuracy', game_metrics.get('gb_accuracy', 0)):.4f}")

In [ ]:
# Task 2: Player Performance Model
print("\nTraining Player Performance Models...")
print("-" * 40)
player_model, player_metrics = train_player_models()
for stat, info in player_metrics.items():
    print(f"  {stat.upper()}: MAE = {info.get('mae', 0):.3f}")

In [ ]:
# Task 3: Playoff Odds Simulation
print("\nRunning Playoff Odds Simulation...")
print("-" * 40)
playoff_df = simulate_playoff_odds()
print(f"\nDone! Simulated {len(playoff_df)} teams")
playoff_df.head(10)

In [ ]:
# Print the final summary
elapsed = datetime.now() - start
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Total time: {elapsed.seconds // 60}m {elapsed.seconds % 60}s")
print("\nModel artifacts saved in models/artifacts/")

In [ ]:
# Download trained model artifacts to your local machine
import shutil
from google.colab import files

shutil.make_archive("/content/model_artifacts", "zip", ".", "models/artifacts")
files.download("/content/model_artifacts.zip")
print("\nExtract the zip into your local project's models/ folder.")

## When you're done

1. Run the cell above to download your trained model artifacts
2. Extract `model_artifacts.zip` into your local project's `models/` folder
3. Click the kernel name in the top right corner of VS Code
4. Choose **Disconnect** to free up the Colab machine